In [65]:
import pandas as pd
from sklearn.feature_selection import mutual_info_classif
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import pickle
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import json

# Load the Dataset

In [66]:
data = pd.read_csv('../preprocessing/cleaned_data.csv')
data = pd.DataFrame(data)  # Placeholder for the actual data cleaning process
data =data.drop(columns=['Gender', 'Segment', 'nps_category', 'last_interaction_date'])

In [67]:
data

,CustomerID,Age,NPS,account_age_days,age_group,total_purchase_value,total_frequency,product_diversity,Plan,subscription_duration_days,subscription_age_days,is_active,total_interactions,total_late_payments,payment_count,late_payment_rate,payment_risk_score,PageViews,TimeSpent(minutes),engagement_ratio,engagement_intensity,total_actions,add_to_cart_count,search_count,click_count,unique_pages,cart_conversion_rate,search_intensity,page_diversity,Logins,frequency_score,engagement_score,avg_rating,avg_comment_length,sentiment_score,is_negative,is_positive,emails_sent,emails_opened,emails_clicked,open_rate,click_rate,click_through_rate,marketing_engagement,ChurnLabel,Recency,Frequency,Monetary,r_score,f_score,m_score,rfm_score,rfm_segment,Gender_Female,Gender_Male,Segment_Segment A,Segment_Segment B,Segment_Segment C,nps_category_Detractor,nps_category_Passive,nps_category_Promoter,days_since_last_interaction
0,1001,31,3,1069,0,3994.72,38,7,2689.622546,871,937,0,4,40,3,10.00,400.00,49,15,0.300000,735,24,8,12,4,13,0.320000,0.480000,0.520000,19,4,76,1.0,96.0,0.2,1,0,8,8,8,0.888889,0.888889,0.888889,0.790123,1,523,722,3994.72,1,4,4,9,4,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,523
1,1002,66,6,1455,2,2844.35,4,3,2828.649426,290,529,0,19,10,3,2.50,25.00,100,9,0.089109,900,24,8,7,9,13,0.320000,0.280000,0.520000,9,4,36,2.0,108.0,0.4,1,0,9,9,9,0.900000,0.900000,0.900000,0.810000,0,17,36,2844.35,5,1,3,9,4,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,17
2,1003,36,3,1341,1,1866.52,14,3,2790.271692,319,1184,0,3,8,3,2.00,16.00,1,97,48.500000,97,12,2,7,3,7,0.153846,0.538462,0.538462,19,1,19,4.0,72.0,0.8,0,1,8,8,8,0.888889,0.888889,0.888889,0.790123,0,360,266,1866.52,1,3,2,6,3,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,360
3,1004,62,1,1033,2,1378.64,28,5,2905.696003,803,1083,0,59,79,3,19.75,1560.25,25,31,1.192308,775,47,15,16,16,14,0.312500,0.333333,0.291667,4,30,120,1.0,78.0,0.2,1,0,10,10,10,0.909091,0.909091,0.909091,0.826446,1,50,112,1378.64,3,2,2,7,4,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,50
4,1005,68,3,1366,2,2425.05,39,6,2714.759176,580,633,0,10,2,3,0.50,1.00,77,51,0.653846,3927,30,17,4,9,12,0.548387,0.129032,0.387097,12,4,48,3.0,99.0,0.6,0,0,7,7,7,0.875000,0.875000,0.875000,0.765625,0,11,468,2425.05,5,3,3,11,2,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12478,13479,55,8,338,2,1196.56,14,3,2790.271692,745,1296,0,10,3,3,0.75,2.25,70,57,0.802817,3990,6,4,1,1,6,0.571429,0.142857,0.857143,22,30,660,2.0,37.0,0.4,1,0,4,4,4,0.800000,0.800000,0.800000,0.640000,0,82,308,1196.56,3,3,2,8,4,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,82
12479,13480,29,7,930,0,710.57,1,1,2859.778143,18,22,0,3,6,3,1.50,9.00,71,66,0.916667,4686,9,3,3,3,8,0.300000,0.300000,0.800000,25,4,100,3.0,102.0,0.6,0,0,7,7,7,0.875000,0.875000,0.875000,0.765625,0,55,25,710.57,3,1,1,5,3,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,55
12480,13481,38,1,809,1,5154.42,63,10,2723.138567,20,546,0,26,83,3,20.75,1722.25,96,1,0.010309,96,26,10,11,5,9,0.370370,0.407407,0.333333,9,1,9,5.0,134.0,1.0,0,1,5,5,5,0.833333,0.833333,0.833333,0.694444,1,22,567,5154.42,4,4,5,13,1,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,22
12481,13482,26,0,920,0,6055.16,58,9,2816.218537,484,894,0,13,67,3,16.75,1122.25,63,2,0.031250,126,38,7,15,16,12,0.179487,0.384615,0.307692,2,1,2,5.0,113.0,1.0,0,1,1,1,1,0.500000,0.500000,0.500000,0.250000,0,45,116,6055.16,3,2,5,10,2,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,45


In [68]:
pd.set_option('display.max_columns', None)  # Show all columns when displaying DataFrame

# Feature Selection

## Mutual Information

In [69]:
# Feature Selection Using Mutual Information

mutual_data = data.copy()

target = mutual_data['ChurnLabel'] 
features = mutual_data.drop(columns=['ChurnLabel', 'CustomerID'], axis = 1) #1 means y-axis or column and 0 means x-axis or row

mi_scores = mutual_info_classif(features, target)

In [70]:
mi_df = pd.DataFrame({
    'Feature': features.columns,
    'Score': mi_scores
})


# Select top features based on mutual information scores
top_features = mi_df[mi_df['Score'] > 0]['Feature'].tolist()

mutual_data = mutual_data[['CustomerID'] + top_features + ['ChurnLabel']]
mutual_data.head()

,CustomerID,Age,NPS,age_group,total_purchase_value,total_frequency,Plan,total_interactions,total_late_payments,payment_count,late_payment_rate,payment_risk_score,TimeSpent(minutes),engagement_ratio,engagement_intensity,total_actions,add_to_cart_count,click_count,cart_conversion_rate,search_intensity,page_diversity,frequency_score,engagement_score,avg_rating,avg_comment_length,sentiment_score,is_negative,is_positive,emails_clicked,open_rate,click_rate,click_through_rate,marketing_engagement,Recency,Monetary,r_score,f_score,m_score,rfm_score,rfm_segment,Gender_Female,Segment_Segment B,nps_category_Detractor,nps_category_Passive,nps_category_Promoter,days_since_last_interaction,ChurnLabel
0,1001,31,3,0,3994.72,38,2689.622546,4,40,3,10.00,400.00,15,0.300000,735,24,8,4,0.320000,0.480000,0.520000,4,76,1.0,96.0,0.2,1,0,8,0.888889,0.888889,0.888889,0.790123,523,3994.72,1,4,4,9,4,0.0,1.0,1.0,0.0,0.0,523,1
1,1002,66,6,2,2844.35,4,2828.649426,19,10,3,2.50,25.00,9,0.089109,900,24,8,9,0.320000,0.280000,0.520000,4,36,2.0,108.0,0.4,1,0,9,0.900000,0.900000,0.900000,0.810000,17,2844.35,5,1,3,9,4,1.0,0.0,1.0,0.0,0.0,17,0
2,1003,36,3,1,1866.52,14,2790.271692,3,8,3,2.00,16.00,97,48.500000,97,12,2,3,0.153846,0.538462,0.538462,1,19,4.0,72.0,0.8,0,1,8,0.888889,0.888889,0.888889,0.790123,360,1866.52,1,3,2,6,3,1.0,1.0,1.0,0.0,0.0,360,0
3,1004,62,1,2,1378.64,28,2905.696003,59,79,3,19.75,1560.25,31,1.192308,775,47,15,16,0.312500,0.333333,0.291667,30,120,1.0,78.0,0.2,1,0,10,0.909091,0.909091,0.909091,0.826446,50,1378.64,3,2,2,7,4,1.0,0.0,1.0,0.0,0.0,50,1
4,1005,68,3,2,2425.05,39,2714.759176,10,2,3,0.50,1.00,51,0.653846,3927,30,17,9,0.548387,0.129032,0.387097,4,48,3.0,99.0,0.6,0,0,7,0.875000,0.875000,0.875000,0.765625,11,2425.05,5,3,3,11,2,1.0,0.0,1.0,0.0,0.0,11,0


In [71]:
mi_df.sort_values(by='Score', ascending=False)


,Feature,Score
15,payment_risk_score,0.576617
14,late_payment_rate,0.576223
12,total_late_payments,0.575824
11,total_interactions,0.286476
17,TimeSpent(minutes),0.269302
1,NPS,0.230215
19,engagement_intensity,0.142309
18,engagement_ratio,0.137821
56,nps_category_Detractor,0.108202
59,days_since_last_interaction,0.064470


## Combining MI and Intuition

In [72]:
# Feature Selection Based on Intuition
# I will select them by removing features with an MI score of less than 0.01
intuitive_features = mi_df[mi_df['Score'] >= 0.01]['Feature'].tolist()
print(len(intuitive_features), "features selected based on intuition.")
intuitive_data = data[['CustomerID'] + intuitive_features + ['ChurnLabel']]
intuitive_data.head()

# Train a Logistic Regression model using only the intuitively selected features
X_intuitive = intuitive_data.drop(columns=['ChurnLabel', 'CustomerID'], axis=1)
y_intuitive = intuitive_data['ChurnLabel']

16 features selected based on intuition.


# Model Training

## Sample Data Creation

In [73]:
# Create a sample data using the first row of mutual_data, each column will randomly add or subtract 10% of its value
sample_data = X_intuitive.iloc[0].copy()
for col in X_intuitive.columns:
    if col != 'CustomerID' and col != 'ChurnLabel':
        variation = sample_data[col] * 0.1
        sample_data[col] += np.random.uniform(-variation, variation)

sample_data = sample_data.values

# Turn to dataframe
sample_df = pd.DataFrame([sample_data], columns=X_intuitive.columns)
sample_df.head()

,NPS,total_interactions,total_late_payments,late_payment_rate,payment_risk_score,TimeSpent(minutes),engagement_ratio,engagement_intensity,Recency,r_score,rfm_score,rfm_segment,nps_category_Detractor,nps_category_Passive,nps_category_Promoter,days_since_last_interaction
0,2.756383,3.66489,43.40261,9.285189,399.524639,14.867231,0.310251,681.562907,504.220815,0.936802,9.392493,3.614777,0.998809,0.0,0.0,528.943509


## Training a Decision Tree Model

In [74]:
def train_and_evaluate_DT(X_intuitive, y_intuitive):
    """This function trains and evaluates a Decision Tree Classifier model using only the features selected based on mutual information scores."""

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X_intuitive, y_intuitive, test_size=0.2, random_state=42, stratify=y_mutual)

    # Define the model
    model = DecisionTreeClassifier(random_state=42) 

    param_dist = {
        'criterion': ['gini', 'entropy', 'log_loss'],
        'max_depth': [100, 200, 300],
        'splitter': ['best', 'random'],
        'min_samples_split': [2, 4, 6],
        'min_samples_leaf': [1, 2, 4],
        'max_features': [0.2, 0.4, 0.6]
    }

    random_search = RandomizedSearchCV(model, param_distributions=param_dist, n_iter=10, scoring='accuracy', cv=4, random_state=42, n_jobs=1)
    random_search.fit(X_train, y_train)
    best_model = random_search.best_estimator_
    # print("Best Hyperparameters:", best_param)
    y_pred = best_model.predict(X_test)

    # Save and Load Model

    # Save model
    with open('model.pkl', 'wb') as f:
        pickle.dump(best_model, f)
    # Load the saved model
    with open('model.pkl', 'rb') as f:
        model = pickle.load(f)
    
    # Save as json
    with open('model.json', 'w') as f:
        import json
        json.dump(X_intuitive.columns.tolist(), f)



    evaluation_metrics = {
        'confusion_matrix': confusion_matrix(y_test, y_pred),
        'classification_report': classification_report(y_test, y_pred),
        'accuracy': accuracy_score(y_test, y_pred)
    }

    print("Best Params:", random_search.best_params_)
    print("Decision Tree Evaluation Metrics:")
    print("Confusion Matrix:")
    print(evaluation_metrics['confusion_matrix'])
    print("Classification Report:")
    print(evaluation_metrics['classification_report'])
    print("Accuracy:", f"{evaluation_metrics['accuracy']:.2f}")

train_and_evaluate_DT(X_intuitive, y_intuitive)


Best Params: {'splitter': 'random', 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': 0.2, 'max_depth': 200, 'criterion': 'entropy'}
Decision Tree Evaluation Metrics:
Confusion Matrix:
[[1207   27]
 [  26 1237]]
Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98      1234
           1       0.98      0.98      0.98      1263

    accuracy                           0.98      2497
   macro avg       0.98      0.98      0.98      2497
weighted avg       0.98      0.98      0.98      2497

Accuracy: 0.98


### Testing the DT Model

In [75]:
def inference(sample):

    # Load the saved model
    with open('model.pkl', 'rb') as f:
        model = pickle.load(f)

    # Make prediction
    prediction = model.predict(sample)
    return f"The churn prediction is: {prediction}"

inference(sample_df)

'The churn prediction is: [1]'

## Training a Logistic Regression Model

In [76]:
def train_and_evaluate_LR(X_intuitive, y_intuitive):
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X_intuitive,
        y_intuitive,
        test_size=0.2,
        random_state=42,
        stratify=y_intuitive
    )

    # Pipeline: scaling + model
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('logreg', LogisticRegression(max_iter=500, random_state=42))
    ])

    # Hyperparameter search space
    param_dist = {
        'logreg__C': np.logspace(-4, 4, 20),
        'logreg__penalty': ['l1', 'l2'],
        'logreg__solver': ['liblinear'],
        'logreg__class_weight': [None, 'balanced']
    }

    random_search = RandomizedSearchCV(
        pipeline,
        param_distributions=param_dist,
        n_iter=20,
        scoring='accuracy',
        cv=4,
        random_state=42,
        n_jobs=-1
    )

    random_search.fit(X_train, y_train)

    best_model = random_search.best_estimator_

    y_pred = best_model.predict(X_test)

    # Save model (PIPELINE, not just estimator)
    with open('logreg_model.pkl', 'wb') as f:
        pickle.dump(best_model, f)

    # Save feature names
    with open('model.json', 'w') as f:
        json.dump(X_mutual.columns.tolist(), f)

    evaluation_metrics = {
        'confusion_matrix': confusion_matrix(y_test, y_pred),
        'classification_report': classification_report(y_test, y_pred),
        'accuracy': accuracy_score(y_test, y_pred)
    }

    print("Best Params:", random_search.best_params_)
    print("Log Regression Evaluation Metrics:")
    print("Confusion Matrix:")
    print(evaluation_metrics['confusion_matrix'])
    print("Classification Report:")
    print(evaluation_metrics['classification_report'])
    print("Accuracy:", f"{evaluation_metrics['accuracy']:.2f}")

    return best_model

train_and_evaluate_LR(X_intuitive, y_intuitive)

Best Params: {'logreg__solver': 'liblinear', 'logreg__penalty': 'l2', 'logreg__class_weight': None, 'logreg__C': np.float64(11.288378916846883)}
Log Regression Evaluation Metrics:
Confusion Matrix:
[[1208   26]
 [  33 1230]]
Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.98      0.98      1234
           1       0.98      0.97      0.98      1263

    accuracy                           0.98      2497
   macro avg       0.98      0.98      0.98      2497
weighted avg       0.98      0.98      0.98      2497

Accuracy: 0.98


c:\Users\bubut\Documents\personalprojects\amdari\REDER-PREDICTIONS\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('logreg', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'l2'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",np.float64(11.288378916846883)
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some p

### Testing the LR Model

In [77]:
def logreg_inference(sample):

    # Load the saved model
    with open('logreg_model.pkl', 'rb') as f:
        model = pickle.load(f)

    # Make prediction
    prediction = model.predict(sample)
    return f"The churn prediction is: {prediction}"

logreg_inference(sample_df)

'The churn prediction is: [1]'